CELL 1: IMPORTS AND SETUP

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine
import os

notebook_dir = os.getcwd()

if "analysis" in notebook_dir:
    db_path = os.path.abspath(os.path.join(notebook_dir, "..", "database", "sales.db"))
else:
    db_path = os.path.abspath(os.path.join(notebook_dir, "database", "sales.db"))

db_path = db_path.replace("\\","/")

#connect to the SQLite database
engine = create_engine(f"sqlite:///{db_path}")

#consistent style for every chart
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12,5)
plt.rcParams["font.size"] = 12

#reads a .sql file and returns the contents as a string
def read_sql_file(filepath):
    with open(filepath, "r") as f:
        return f.read()

#saves charts to analysis/charts/ folder and displays it
def save_and_show(filename):
    os.makedirs("analysis/charts", exist_ok=True)
    plt.savefig(f"analysis/charts/{filename}", dpi=150, bbox_inches="tight")
    plt.show()

print("Setup complete - database connected")

Setup complete - database connected


CELL 2: KPI SUMMARY 

In [2]:
total_revenue = pd.read_sql(
    "SELECT SUM(revenue) FROM sales_clean", engine).iloc[0,0]

total_orders = pd.read_sql(
    "SELECT COUNT(order_id) FROM sales_clean", engine).iloc[0,0]

total_lost = pd.read_sql(
    "SELECT SUM(lost_revenue) FROM sales_clean", engine).iloc[0,0]

total_days = pd.read_sql(
    "SELECT MAX(day_number) FROM sales_clean", engine).iloc[0,0]

print("E-COMMERCE PIPELINE - 30 DAY KPI SUMMARY")
print(f"Days of data: {int(total_days)}")
print(f"Total orders: {int(total_orders):,}")
print(f"Total revenue earned: Rs.{total_revenue:,.0f}")
print(f"Total revenue lost: Rs.{total_lost:,.0f}")

E-COMMERCE PIPELINE - 30 DAY KPI SUMMARY
Days of data: 30
Total orders: 1,202
Total revenue earned: Rs.5,407,573
Total revenue lost: Rs.1,115,274


CELL 3: DATASET OVERVIEW

In [3]:
df = pd.read_sql("SELECT * FROM sales_clean", engine)

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values per column:\n{df.isnull().sum()}")
print(f"\nBasic statistics for numberic columns:")
df[["price", "quantity", "fulfilled", "revenue", "lost_revenue", "stock_remaining"]].describe().round(2)

Dataset shape: 1,202 rows x 18 columns

Column names:
['order_id', 'date', 'day_number', 'product_id', 'product_name', 'category', 'price', 'quantity', 'fulfilled', 'revenue', 'city', 'payment_method', 'customer_name', 'customer_phone', 'stock_remaining', 'lost_sales', 'lost_revenue', 'phone_valid']

Data types:
order_id             str
date                 str
day_number         int64
product_id           str
product_name         str
category             str
price              int64
quantity           int64
fulfilled          int64
revenue            int64
city                 str
payment_method       str
customer_name        str
customer_phone       str
stock_remaining    int64
lost_sales         int64
lost_revenue       int64
phone_valid        int64
dtype: object

Missing values per column:
order_id           0
date               0
day_number         0
product_id         0
product_name       0
category           0
price              0
quantity           0
fulfilled          0
reven

,price,quantity,fulfilled,revenue,lost_revenue,stock_remaining
count,1202.00,1202.00,1202.00,1202.00,1202.00,1202.00
mean,965.47,5.27,4.37,4498.81,927.85,123.24
std,618.68,3.40,3.77,7522.55,3288.62,97.04
min,299.00,2.00,0.00,0.00,0.00,0.00
25%,499.00,3.00,2.00,1396.00,0.00,32.25
50%,709.00,5.00,4.00,2895.00,0.00,118.00
75%,1189.00,7.00,6.00,5593.00,0.00,198.00
max,2199.00,68.00,68.00,149532.00,54975.00,406.00


CELL 4: ETL DATA QUALITY REPORT

In [5]:
total_rows = len(df)

unknown_names = len(df[df["customer_name"] == "Unknown"])
unknown_phones = len(df[df["customer_phone"] == "Unknown"])
unknown_cities = len(df[df["city"] == "Unknown"])

invalid_phones = len(df[df["phone_valid"] == 0])

print("ETL PIPELINE - DATA QUALITY REPORT")
print(f"\nTotal rows processed: {total_rows:,}")
print()
print(f"Missing customer names: {unknown_names}")
print(f"Missing phone numbers: {unknown_phones}")
print(f"Missing city values: {unknown_cities}")
print(f"Invalid phone numbers: {invalid_phones}")

ETL PIPELINE - DATA QUALITY REPORT

Total rows processed: 1,202

Missing customer names: 90
Missing phone numbers: 85
Missing city values: 99
Invalid phone numbers: 127
